# Episode 9 — Risk II: bucketed delta

Companion notebook for the video. One DV01 hides *where* along the curve the risk sits. We split a 4-year swap's risk by quote (par bucket delta) and by the Basel FRTB tenors on each curve (zero bucket delta), hedge it with two standard swaps, and see what's left: basis risk. Every number on screen or in the narration is produced here and read from `build/outputs.json`.

> **Illustrative data.** The quotes in `quotes_illustrative.csv` (as in Episode 6) are made up for teaching. They are **not market prices**. Educational material only, not investment advice. The FRTB tenors come from the Basel Framework (MAR21.8); this notebook is not a capital calculation.

1. Setup · 2. The trade · 3. Par bucket delta · 4. FRTB zero buckets · 5. The hedge · 6. What's left · 7. Export

**Running in Google Colab?** Run the next cells first: they install QuantLib (version 1.43, the one used in the video) and write the data file this notebook reads. Then run the rest of the notebook in order.

In [ ]:
# Colab doesn't include QuantLib. Install it (about 30 seconds).
# The video used QuantLib 1.43; drop '==1.43' for the latest.
!pip install QuantLib==1.43

In [ ]:
#@title Data: writes `quotes_illustrative.csv` (run me first) { display-mode: "form" }
# Illustrative quotes, made up for teaching. Not market data.
from pathlib import Path
Path('quotes_illustrative.csv').parent.mkdir(parents=True, exist_ok=True)
Path('quotes_illustrative.csv').write_text("""curve,instrument,tenor,value,unit
AONIA,deposit,O/N,3.85,pct
AONIA,OIS,1M,3.86,pct
AONIA,OIS,3M,3.89,pct
AONIA,OIS,6M,3.93,pct
AONIA,OIS,9M,3.97,pct
AONIA,OIS,1Y,4.00,pct
AONIA,OIS,18M,4.05,pct
AONIA,OIS,2Y,4.08,pct
AONIA,OIS,3Y,4.13,pct
AONIA,OIS,5Y,4.24,pct
AONIA,OIS,7Y,4.35,pct
AONIA,OIS,10Y,4.50,pct
BBSW3M,fixing,3M,4.02,pct
BBSW3M,AONIA/BBSW basis,1Y,13.0,bp
BBSW3M,AONIA/BBSW basis,2Y,14.0,bp
BBSW3M,AONIA/BBSW basis,3Y,15.0,bp
BBSW3M,AONIA/BBSW basis,5Y,15.5,bp
BBSW3M,AONIA/BBSW basis,7Y,16.0,bp
BBSW3M,AONIA/BBSW basis,10Y,16.5,bp
BBSW6M,fixing,6M,4.14,pct
BBSW6M,3s6s basis,1Y,8.0,bp
BBSW6M,3s6s basis,2Y,9.0,bp
BBSW6M,3s6s basis,3Y,10.0,bp
BBSW6M,3s6s basis,5Y,11.0,bp
BBSW6M,3s6s basis,7Y,11.5,bp
BBSW6M,3s6s basis,10Y,12.0,bp
""")
print('wrote quotes_illustrative.csv')

In [ ]:
# Parameters (papermill overrides these)
valuation_date = "2026-09-22"
quotes_file = "quotes_illustrative.csv"
output_json = "build/outputs.json"
notional = 100_000_000
tenor = "4Y"

## 1. Setup

In [ ]:
import json
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import QuantLib as ql

today = ql.DateParser.parseISO(valuation_date)
ql.Settings.instance().evaluationDate = today
dc = ql.Actual365Fixed()
iso = lambda d: d.ISO()
cal = ql.Australia(ql.Australia.Settlement)
# QuantLib 1.43 misses NSW's additional Anzac Day holidays when 25 April falls on a weekend.
for d in [ql.Date(27, 4, 2026), ql.Date(26, 4, 2027)]:
    cal.addHoliday(d)

out = {"meta": {
    "episode": 9, "valuation_date": valuation_date, "quantlib_version": ql.__version__,
    "quotes_label": "ILLUSTRATIVE - not market data",
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}
print("QuantLib", ql.__version__, "| valuation date", today)

The curve-building code shared by Episodes 6 to 9:

In [ ]:
# AUD curve family used from Episode 6 on. Copied verbatim into each notebook by make_notebook.py,
# so every notebook runs on its own. Needs: ql, pd, today, cal, dc (defined in the setup cell).

def aonia_index(curve=ql.YieldTermStructureHandle()):
    return ql.OvernightIndex("AONIA", 0, ql.AUDCurrency(), cal, dc, curve)

def bbsw(months, curve=ql.YieldTermStructureHandle()):
    # Set on the first day of each period (no fixing lag), Modified Following, no end-of-month rule.
    return ql.IborIndex(f"BBSW{months}M", ql.Period(months, ql.Months), 0, ql.AUDCurrency(),
                        cal, ql.ModifiedFollowing, False, dc, curve)

def build_curves(quotes):
    """AONIA from OIS quotes; 3M BBSW = AONIA + AONIA/BBSW basis; 6M BBSW = 3M BBSW + 3s6s basis.

    Returns the curves, their handles and the SimpleQuote behind every input, keyed (curve, tenor).
    Changing a quote with setValue() flows through all three curves.
    """
    q = {}
    def handle(row):
        scale = 1e4 if row.unit == "bp" else 100
        q[(row.curve, row.tenor)] = ql.SimpleQuote(row.value / scale)
        return ql.QuoteHandle(q[(row.curve, row.tenor)])

    rows = lambda curve: quotes[quotes.curve == curve].itertuples()
    MF = ql.ModifiedFollowing

    ois_helpers = []
    for r in rows("AONIA"):
        if r.instrument == "deposit":
            h = ql.DepositRateHelper(handle(r), ql.Period(1, ql.Days), 0, cal,
                                     ql.Following, False, dc)
        else:
            h = ql.OISRateHelper(1, ql.Period(r.tenor), handle(r), aonia_index(), paymentLag=2,
                                 paymentFrequency=ql.Annual, paymentCalendar=cal,
                                 convention=MF, endOfMonth=False)
        ois_helpers.append(h)
    aonia = ql.PiecewiseLogLinearDiscount(today, ois_helpers, dc)
    aonia.enableExtrapolation()
    aonia_h = ql.YieldTermStructureHandle(aonia)

    h3 = []
    for r in rows("BBSW3M"):
        if r.instrument == "fixing":
            h3.append(ql.DepositRateHelper(handle(r), bbsw(3)))
        else:  # AONIA + spread vs 3M BBSW, both quarterly
            h3.append(ql.OvernightIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                aonia_index(aonia_h), bbsw(3), aonia_h))
    bbsw3m = ql.PiecewiseLogLinearDiscount(today, h3, dc)
    bbsw3m.enableExtrapolation()
    bbsw3m_h = ql.YieldTermStructureHandle(bbsw3m)

    h6 = []
    for r in rows("BBSW6M"):
        if r.instrument == "fixing":
            h6.append(ql.DepositRateHelper(handle(r), bbsw(6)))
        else:  # 3M BBSW + spread (quarterly) vs 6M BBSW (semi-annual)
            h6.append(ql.IborIborBasisSwapRateHelper(
                handle(r), ql.Period(r.tenor), 1, cal, MF, False,
                bbsw(3, bbsw3m_h), bbsw(6), aonia_h, False))
    bbsw6m = ql.PiecewiseLogLinearDiscount(today, h6, dc)
    bbsw6m.enableExtrapolation()

    helpers = {"AONIA": ois_helpers, "BBSW3M": h3, "BBSW6M": h6}
    curves = {"AONIA": aonia, "BBSW3M": bbsw3m, "BBSW6M": bbsw6m}
    for c in curves.values():
        c.nodes()  # bootstrap now, in order
    handles = {k: ql.YieldTermStructureHandle(c) for k, c in curves.items()}
    return curves, handles, q, helpers

def vanilla_swap(tenor, fixed_rate, index_months, forecast, discount, notional,
                 receive=True, start=None):
    """AUD vanilla swap: quarterly vs 3M BBSW or semi-annual vs 6M BBSW, ACT/365F, T+1 start."""
    start = start or cal.advance(today, 1, ql.Days)
    end = cal.advance(start, ql.Period(tenor), ql.ModifiedFollowing, False)
    freq = ql.Quarterly if index_months == 3 else ql.Semiannual
    sched = ql.Schedule(start, end, ql.Period(freq), cal, ql.ModifiedFollowing,
                        ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
    side = ql.VanillaSwap.Receiver if receive else ql.VanillaSwap.Payer
    index = bbsw(index_months, forecast)
    sw = ql.VanillaSwap(side, notional, sched, fixed_rate, dc, sched, index, 0.0, dc)
    sw.setPricingEngine(ql.DiscountingSwapEngine(discount))
    return sw

## 2. The trade, and curves we can bump

Receive fixed at par on a **4-year** swap against 6-month BBSW. Four years is between two quoted tenors (3Y and 5Y), which is where bucketing gets interesting.

Each curve is wrapped in a piecewise zero spread at the Basel FRTB tenors (MAR21.8), all zero for now. Moving one of those spreads by 1bp moves the zero curve by 1bp at that tenor, tapering linearly to zero at the neighbouring tenors: a triangular bucket.

In [ ]:
quotes = pd.read_csv(quotes_file)
curves, H, quote_handles, helpers = build_curves(quotes)

FRTB_TENORS = [0.25, 0.5, 1, 2, 3, 5, 10, 15, 20, 30]         # years, Basel MAR21.8
vertex_dates = [today + ql.Period(round(t * 12), ql.Months) for t in FRTB_TENORS]
zq = {c: [ql.SimpleQuote(0.0) for _ in FRTB_TENORS] for c in curves}
W = {c: ql.YieldTermStructureHandle(ql.PiecewiseZeroSpreadedTermStructure(
         H[c], [ql.QuoteHandle(q) for q in zq[c]], vertex_dates)) for c in curves}
for c in W:
    W[c].enableExtrapolation()

def swap_on_wrapped(t, rate, months, receive):
    return vanilla_swap(t, rate, months, W[f"BBSW{months}M"], W["AONIA"], notional, receive=receive)

par4 = swap_on_wrapped(tenor, 0.04, 6, True).fairRate()
trade = swap_on_wrapped(tenor, par4, 6, True)
out["trade"] = {"tenor": tenor, "notional": notional, "par_pct": par4 * 100, "npv": trade.NPV()}
pd.Series(out["trade"])

## 3. Par bucket delta

Move one quote at a time by ±1bp, let the curves rebuild, reprice. We report the change in value for **+1bp**, the sign convention FRTB uses (MAR21.19).

In [ ]:
def par_deltas(instr):
    rows = []
    for (curve, tnr), q in quote_handles.items():
        q0 = q.value()
        q.setValue(q0 + 1e-4); up = instr.NPV()
        q.setValue(q0 - 1e-4); down = instr.NPV()
        q.setValue(q0)
        rows.append({"curve": curve, "tenor": tnr, "delta": (up - down) / 2})
    return pd.DataFrame(rows)

GROUP = {"AONIA": "AONIA swaps", "BBSW3M": "AONIA/BBSW basis", "BBSW6M": "3s6s basis"}
d4 = par_deltas(trade)
d4["group"] = d4.curve.map(GROUP)
d4.loc[d4.tenor.isin(["3M", "6M"]) & d4.curve.isin(["BBSW3M", "BBSW6M"]), "group"] = "BBSW fixings"
big = d4[d4.delta.abs() > 1000].copy()   # the buckets that matter
big["label"] = big.group + " " + big.tenor
out["par_buckets"] = {"rows": big[["label", "delta"]].to_dict("records"),
                      "total_rates": float(d4[d4.curve == "AONIA"].delta.sum()),
                      "rates_3y": float(d4[(d4.curve == "AONIA") & (d4.tenor == "3Y")].delta.iloc[0]),
                      "rates_5y": float(d4[(d4.curve == "AONIA") & (d4.tenor == "5Y")].delta.iloc[0])}
out["par_buckets"]["share_5y"] = out["par_buckets"]["rates_5y"] / out["par_buckets"]["total_rates"]
big

## 4. FRTB zero buckets

Bump the zero-spread at each FRTB tenor, one curve at a time. Basel treats AONIA, 3-month BBSW and 6-month BBSW as three separate curves (MAR21.8). The buckets of one curve add up to a parallel 1bp shift of that curve.

In [ ]:
def zero_deltas(instr):
    table = {}
    for c in ("AONIA", "BBSW3M", "BBSW6M"):
        col = []
        for q in zq[c]:
            q.setValue(1e-4); up = instr.NPV()
            q.setValue(-1e-4); down = instr.NPV()
            q.setValue(0.0)
            col.append((up - down) / 2)
        table[c] = col
    return pd.DataFrame(table, index=FRTB_TENORS)

z4 = zero_deltas(trade)
parallel = {}
for c in z4.columns:
    for q in zq[c]:
        q.setValue(1e-4)
    up = trade.NPV()
    for q in zq[c]:
        q.setValue(-1e-4)
    down = trade.NPV()
    for q in zq[c]:
        q.setValue(0.0)
    parallel[c] = (up - down) / 2
shown = z4.loc[[t for t in FRTB_TENORS if t <= 10]]
out["zero_buckets"] = {
    "rows": [{"tenor": ("3M" if t == 0.25 else "6M" if t == 0.5 else f"{int(t)}Y"),
              "aonia": r.AONIA, "bbsw3m": r.BBSW3M, "bbsw6m": r.BBSW6M} for t, r in shown.iterrows()],
    "sum_6m": float(z4.BBSW6M.sum()), "parallel_6m": parallel["BBSW6M"],
    "sum_aonia": float(z4.AONIA.sum()), "parallel_aonia": parallel["AONIA"],
    "b6m_3y": float(z4.loc[3, "BBSW6M"]), "b6m_5y": float(z4.loc[5, "BBSW6M"]),
}
z4.round(0)

## 5. The hedge

AFMA's standard swaps are quarterly against 3-month BBSW out to three years and semi-annual against 6-month BBSW from four. So we hedge with a **3-year quarterly** swap and a **5-year semi-annual** swap, both paying fixed at par, sized so the portfolio has no risk to the 3-year and 5-year AONIA quotes.

In [ ]:
hedge3 = swap_on_wrapped("3Y", swap_on_wrapped("3Y", 0.04, 3, False).fairRate(), 3, False)
hedge5 = swap_on_wrapped("5Y", swap_on_wrapped("5Y", 0.04, 6, False).fairRate(), 6, False)
d3, d5 = par_deltas(hedge3), par_deltas(hedge5)
key = lambda d, t: float(d[(d.curve == "AONIA") & (d.tenor == t)].delta.iloc[0])
A = np.array([[key(d3, "3Y"), key(d5, "3Y")], [key(d3, "5Y"), key(d5, "5Y")]])
b = -np.array([key(d4, "3Y"), key(d4, "5Y")])
w3, w5 = np.linalg.solve(A, b)                 # multiples of AUD 100m, paying fixed
out["hedge"] = {"rows": [
    {"instrument": "Pay fixed 3Y, quarterly vs 3M BBSW", "notional": w3 * notional},
    {"instrument": "Pay fixed 5Y, semi-annual vs 6M BBSW", "notional": w5 * notional}],
    "w3": w3, "w5": w5}
pd.DataFrame(out["hedge"]["rows"])

## 6. What's left

Add up the par bucket deltas by group, before and after the hedge. The rate risk is gone; basis risk is not, because the 3-year hedge pays 3-month BBSW while the trade pays 6-month.

In [ ]:
port = d4.copy()
port["hedges"] = w3 * d3.delta.values + w5 * d5.delta.values
port["net"] = port.delta + port.hedges
groups = port.groupby("group")[["delta", "hedges", "net"]].sum()
order = ["AONIA swaps", "AONIA/BBSW basis", "3s6s basis"]
groups = groups.reindex(order).fillna(0)
out["residual"] = {"rows": [{"group": g, "trade": r.delta, "hedges": r.hedges, "net": r.net} for g, r in groups.iterrows()],
                   "rates_net": float(groups.loc["AONIA swaps", "net"]),
                   "b36_net": float(groups.loc["3s6s basis", "net"]),
                   "bob_net": float(groups.loc["AONIA/BBSW basis", "net"])}
b36 = port[port.curve == "BBSW6M"][["tenor", "delta", "hedges", "net"]]
out["residual"]["b36_rows"] = b36[b36.net.abs() > 50].to_dict("records")

# A 3-year 3s6s basis swap (receive 6M, pay 3M + spread) to cancel the leftover basis risk.
end3 = cal.advance(cal.advance(today, 1, ql.Days), ql.Period("3Y"), ql.ModifiedFollowing, False)
s3 = ql.Schedule(cal.advance(today, 1, ql.Days), end3, ql.Period(ql.Quarterly), cal, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
s6 = ql.Schedule(cal.advance(today, 1, ql.Days), end3, ql.Period(ql.Semiannual), cal, ql.ModifiedFollowing, ql.ModifiedFollowing, ql.DateGeneration.Forward, False)
spread3 = quote_handles[("BBSW6M", "3Y")].value()
basis3 = ql.Swap(ql.IborLeg([notional], s3, bbsw(3, W["BBSW3M"]), dc, spreads=[spread3]),
                 ql.IborLeg([notional], s6, bbsw(6, W["BBSW6M"]), dc))
basis3.setPricingEngine(ql.DiscountingSwapEngine(W["AONIA"]))
db = par_deltas(basis3)
per_unit = float(db[db.curve == "BBSW6M"].delta.sum())
wb = -out["residual"]["b36_net"] / per_unit
out["basis_hedge"] = {"notional": wb * notional, "side": "receive 6M, pay 3M + spread" if wb > 0 else "pay 6M, receive 3M + spread",
                      "abs_notional": abs(wb) * notional}
groups

## 7. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")